# WildFusion 🦙 🧬

This notebook presents an improved baseline for individual identification using the **[MegaDescriptor](https://arxiv.org/pdf/2311.09118)** with **[ALIKED](https://arxiv.org/pdf/2304.03608)** methods, combined using [WildFusion](https://arxiv.org/pdf/2408.12934).

## What is WildFusion?  
WildFusion is a **feature fusion method** designed to improve the accuracy and robustness of models in individual recognition tasks. Traditional identification models often rely on a single type of feature representation, which may struggle with variations in lighting, angles, and occlusions. WildFusion overcomes these challenges by combining multiple feature extraction techniques, resulting in a more comprehensive and adaptable approach.  

## Dependencies instalation
For the competition we provide two Python packages for loading and preprocessing of available datasets ([wildlife-datasets](https://github.com/WildlifeDatasets/wildlife-datasets)) and tools / method for animal re-identification ([wildlife-tools](https://github.com/WildlifeDatasets/wildlife-tools)).

In [ ]:
# !pip install -q git+https://github.com/WildlifeDatasets/wildlife-datasets@develop
# !pip install -q git+https://github.com/WildlifeDatasets/wildlife-tools
# !pip install -q git+https://github.com/WildMeOrg/wbia-plugin-miew-id.git
# !pip install lmdb
# !pip install -q "transformers==4.45.2" timm albumentations scikit-learn pandas numpy torch networkx

# Imports

In [ ]:
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from wildlife_datasets.datasets import AnimalCLEF2026

from src.feature_extraction.wildfusion import WildFusionFeatureExtractor
from src.clustering_algorithms.chinese_whispers import chinese_whispers
from src.utilities.image2identity import *
from src.utilities.reranking import re_ranking

# Setups

In [ ]:
# Setup Paths and Metadata
root = 'dataset root'
sample_sub_path = f'{root}/sample_submission.csv'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

#Loading the 2026 dataset
dataset = AnimalCLEF2026(root, load_label=True)

# Splits
dataset_database = dataset.get_subset(dataset.metadata['split'] == 'train')
dataset_query = dataset.get_subset(dataset.metadata['split'] == 'test')

print(f"Database (Train) size: {len(dataset_database)}")
print(f"Test (Query) size: {len(dataset_query)}")

In [ ]:
# Use a subset for calibration (Generic/No species-specific prep during global fit)
dataset_calibration = dataset_database.get_subset(range(min(512, len(dataset_database)))) 

wildfusion = WildFusionFeatureExtractor(dataset_calibration, 
                                        device=device, 
                                        sphereface_version="check point path to the sphereface2 model")
wildfusion.calibrate()

In [ ]:
best_configs = {
    "LynxID2025": {
        "top_r": 1,
        "tau_attach": 0.28,
        "tau_margin": 0.42000000000000004,
        "threshold": 0.29000000000000004,
        "top_k": 42,
        "iterations": 10,
        "k_core_degree": 1,
        "use_rerank": True,
        "rr_k1": 10,
        "rr_k2": 1,
        "rr_lambda": 0.8076405888420517,
    },
    "SalamanderID2025": {
        "top_r": 2,
        "tau_attach": 0.35,
        "tau_margin": 0.23,
        "threshold": 0.4,
        "top_k": 60,
        "iterations": 20,
        "k_core_degree": 5,
        "use_rerank": True,
        "rr_k1": 7,
        "rr_k2": 4,
        "rr_lambda": 0.7182900346367315,
    },
    "SeaTurtleID2022": {
        "top_r": 5,
        "tau_attach":  0.8200000000000001,
        "tau_margin": 0.45,
        "threshold": 0.22999999999999998,
        "top_k": 47,
        "iterations": 25,
        "k_core_degree": 1,
        "use_rerank": True,
        "rr_k1": 20,
        "rr_k2": 5,
        "rr_lambda": 0.8518428732278848,
    },
    "TexasHornedLizards": {
        "threshold": 0.49,
        "top_k": 15,
        "iterations": 20,
        "k_core_degree": 1,
        "use_rerank": False,
    },
}

RR_SPECIES = {"LynxID2025", "SalamanderID2025", "SeaTurtleID2022"}
results = []

# MAIN LOOP


In [ ]:
CACHE = True

# Process each species/dataset separately so every dataset can use its own
# clustering and re-ranking configuration.
for ds_name in dataset.metadata['dataset'].unique():
    print(f"\n--- Processing Dataset: {ds_name} ---")

    # Load the best tuned config for this dataset. If no config exists, use a
    # safe default configuration.
    config = best_configs.get(ds_name, {
        "threshold": 0.49,
        "top_k": 15,
        "iterations": 20,
        "k_core_degree": 1,
        "use_rerank": False,
    })
    print("Config:", config)

    # Select query/test images and database/train images that belong to the
    # current dataset/species.
    ds_test = dataset_query.get_subset(
        dataset_query.metadata['dataset'] == ds_name
    )
    ds_train = dataset_database.get_subset(
        dataset_database.metadata['dataset'] == ds_name
    )

    # Skip this dataset if it has no query images.
    if len(ds_test) == 0:
        continue

    # Compute test-vs-test similarities. These are used for clustering the query
    # images into identities.
    sim_test_test = wildfusion.run(ds_name, data=(ds_test, ds_test))
    sim_test_test = np.clip(sim_test_test, 0.0, 1.0)

    if CACHE:
        save_dir = Path("cache")
        save_dir.mkdir(parents=True, exist_ok=True)
        np.save(f"cache/sim_test_test_{ds_name}.npy", sim_test_test)

    # If known training identities exist for this dataset, compute test-vs-train
    # similarities so query clusters can be attached to known identities.
    if len(ds_train) > 0:
        sim_test_train = wildfusion.run(ds_name, data=(ds_test, ds_train))
        sim_test_train = np.clip(sim_test_train, 0.0, 1.0)
        train_ids = ds_train.metadata["identity"].values

        # Convert image-level test-vs-train scores into identity-level scores.
        identity_names, identity_scores = compute_identity_scores(
            sim_test_train=sim_test_train,
            train_ids=train_ids,
            top_r=config["top_r"]
        )

        # For each test image, get the best known identity, its score, the
        # second-best score, and the confidence margin.
        best_identity_names, best_scores, second_best_scores, margins = summarize_best_identities(
            identity_names, identity_scores
        )
        if CACHE:
            np.save(f"cache/sim_test_train_{ds_name}.npy", sim_test_train)
    else:
        # If there are no known train identities, all query images will be
        # handled as new/unknown clusters.
        sim_test_train = None
        train_ids = None
        best_identity_names = None
        best_scores = np.zeros(len(ds_test), dtype=np.float32)
        margins = np.zeros(len(ds_test), dtype=np.float32)

    # Optionally refine the test-vs-test similarity matrix using k-reciprocal
    # re-ranking before clustering.
    if config.get("use_rerank", False) and ds_name in RR_SPECIES:
        print(
            "Applying k-reciprocal re-ranking:",
            {
                "k1": config["rr_k1"],
                "k2": config["rr_k2"],
                "lambda": config["rr_lambda"],
            }
        )
        sim_for_cw = re_ranking(
            sim_test_test,
            k1=config["rr_k1"],
            k2=config["rr_k2"],
            lambda_value=config["rr_lambda"],
        )
    else:
        sim_for_cw = sim_test_test

    # Cluster query images using Chinese Whispers on the selected similarity
    # matrix.
    print("Running Clustering...")
    cluster_ids = chinese_whispers(
        sim_for_cw,
        threshold=config["threshold"],
        iterations=config["iterations"],
        top_k=config["top_k"],
        k_core_degree=config["k_core_degree"],
        seed=42
    )
    print("Clusters:", len(np.unique(cluster_ids)))

    # Store votes from test images that confidently match known train identities.
    # Structure: cluster_id -> known_identity -> accumulated confidence score.
    cluster_votes = defaultdict(lambda: defaultdict(float))

    if train_ids is not None:
        for i in range(len(ds_test)):
            c = int(cluster_ids[i])

            # Attach a test image to a known identity only if both the best score
            # and the margin over the second-best identity are strong enough.
            if (best_scores[i] >= config["tau_attach"]) and (margins[i] >= config["tau_margin"]):
                full_id = str(best_identity_names[i])
                short_id = full_id.split("_")[-1]
                cluster_votes[c][short_id] += float(best_scores[i])

    # Assign a known-identity label to each cluster using the strongest total
    # vote inside that cluster.
    cluster_to_label = {}
    for c, votes in cluster_votes.items():
        if votes:
            best_label = max(votes.items(), key=lambda kv: kv[1])[0]
            cluster_to_label[c] = f"cluster_{ds_name}_{best_label}"

    # Build the final submission rows. Clusters without confident known-identity
    # votes are labeled as new identities.
    for i in range(len(ds_test)):
        c = int(cluster_ids[i])
        image_id = ds_test.metadata.iloc[i]["image_id"]

        final_label = cluster_to_label.get(
            c,
            f"cluster_{ds_name}_new_{c}"
        )

        results.append({
            "image_id": image_id,
            "cluster": final_label
        })

# Final submission


In [ ]:
sample_sub = pd.read_csv(sample_sub_path)
submission_results = pd.DataFrame(results)

final_submission = sample_sub[["image_id"]].merge(
    submission_results,
    on="image_id",
    how="left"
)

final_submission["cluster"] = final_submission["cluster"].fillna("cluster_unknown_0")
final_submission.to_csv("submission_sphereface.csv", index=False)

print("\n--- FINAL CHECK ---")
print("Rows:", len(final_submission))
print(final_submission.head())
print("Missing cluster labels:", final_submission["cluster"].isna().sum())
print(
    "Labels not starting with cluster_:",
    (~final_submission["cluster"].astype(str).str.startswith("cluster_")).sum()
)
print("Duplicate image_ids:", final_submission["image_id"].duplicated().sum())
print("Unique clusters:", final_submission["cluster"].nunique())